# سبق 12 - ایجنٹ اسکریچ پیڈ کے ساتھ چیٹ ہسٹری کی کمی

یہ نوٹ بک دکھاتی ہے کہ مائیکروسافٹ ایجنٹ فریم ورک استعمال کرتے ہوئے طویل بات چیت میں سیاق و سباق کو کیسے منظم کیا جائے۔ جیسے جیسے بات چیت بڑھتی ہے، ٹوکن کی تعداد بڑھ جاتی ہے — جو آخر کار ماڈل کی سیاق و سباق کی ونڈو سے تجاوز کر جاتی ہے۔ ہم اس کا حل ایک **سیاق و سباق کے خلاصہ کاری کے نمونے** اور ایک **ایجنٹ اسکریچ پیڈ** کے ذریعے کرتے ہیں جو مستقل یادداشت کے لیے ہوتا ہے۔

## آپ کیا سیکھیں گے:
1. **سیاق و سباق کے انتظام کی اہمیت**: ٹوکن کی حد اور سیاق و سباق کی ونڈوز کو سمجھنا
2. **سیاق و سباق سے آگاہ ایجنٹس**: ایسے ایجنٹس بنانا جو اپنی بات چیت کے سیاق و سباق کا انتظام کرتے ہیں
3. **سیاق و سباق کے خلاصہ کاری کا نمونہ**: بات چیت کی تاریخ کو مختصر کرنے کے لیے آلات کا استعمال
4. **ایجنٹ اسکریچ پیڈ**: مستقل یادداشت جو سیاق و سباق کی کمی کے بعد بھی باقی رہتی ہے

## ضروریات:
- Azure OpenAI سیٹ اپ جس میں ماحول کے متغیرات ترتیب دیے گئے ہوں
- پچھلے اسباق سے بنیادی ایجنٹ تصورات کی سمجھ


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## کیوں کانٹیکسٹ مینیجمنٹ اہم ہے

ہر LLM کے پاس ایک محدود **کانٹیکسٹ ونڈو** ہوتی ہے — زیادہ سے زیادہ ٹوکنز کی تعداد جو وہ ایک درخواست میں پروسیس کر سکتا ہے۔ جیسے جیسے کثیر دورانیے کی گفتگو آگے بڑھتی ہے:

- **ٹوکن کی گنتی ہر صارف کے پیغام اور معاون کے جواب کے ساتھ خطی طور پر بڑھتی ہے۔**
- **پرومپٹ ٹوکنز لاگت پر غلبہ رکھتے ہیں** کیونکہ پوری تاریخ ہر بار دوبارہ بھیجی جاتی ہے۔
- آخرکار گفتگو **کانٹیکسٹ ونڈو سے تجاوز کر جاتی ہے** اور ماڈل یا تو کٹاؤ کرتا ہے یا خرابی پیدا ہوتی ہے۔

### کانٹیکسٹ مینیج کرنے کی حکمت عملی

| حکمت عملی | یہ کیسے کام کرتی ہے | تلافی |
|---|---|---|
| **کٹاؤ** | پرانے پیغامات کو ہٹا دینا | ابتدائی کانٹیکسٹ ضائع ہو جاتا ہے |
| **خلاصہ سازی** | پرانے پیغامات کو خلاصے میں سمیٹنا | کچھ تفصیل ضائع ہوتی ہے، لیکن اہم نکات برقرار رہتے ہیں |
| **اسکریچ پیڈ / بیرونی میموری** | گفتگو کے باہر اہم حقائق کو ذخیرہ کرنا | ٹول کالز کی ضرورت پڑتی ہے، لیکن کسی بھی کمی برداشت کر لیتا ہے |

اس نوٹ بک میں ہم **خلاصہ سازی** کو **اسکریچ پیڈ ٹول** کے ساتھ ملا کر استعمال کرتے ہیں تاکہ ایجنٹ بات چیت کی تسلسل کو برقرار رکھ سکے، چاہے گفتگو کی تاریخ مختصر بھی کی جائے۔


## سیاق و سباق سے آگاہ ایجنٹ بنانا


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## طویل گفتگو کی نقل سازی

آئیے ایک کثیر-موڑ گفتگو کے ذریعے چلتے ہیں تاکہ یہ دیکھا جا سکے کہ سیاق و سباق کیسے جمع ہوتا ہے۔ نمائندہ کو چاہیے کہ وہ موڑ کے دوران اہم تفصیلات (پسندیدگیاں، بجٹ، سفر کی تاریخیں) کو برقرار رکھے اور تسلسل کا مظاہرہ کرے۔


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

نوٹ کریں کہ ایجنٹ کیسے پچھلے مکالمات سے سیاق و سباق کو برقرار رکھتا ہے — اسے جاپان، سُشی، مندروں، فوٹو گرافی، $3000 بجٹ، اکیلے سفر، اور اپریل کے وقت کے بارے میں علم ہے۔ ایک مختصر گفتگو میں یہ اچھا کام کرتا ہے، لیکن جیسے جیسے گفتگو بڑھتی ہے، مکمل تاریخ دوبارہ بھیجنا مہنگا ہو جاتا ہے۔

آئیے زیادہ موڑوں کے ساتھ گفتگو جاری رکھیں تاکہ سیاق و سباق کے جمع ہونے کو دیکھا جا سکے:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## سیاق و سباق کا خلاصہ کرنے کا نمونہ

جیسا کہ گفتگو بڑھتی ہے، ہم ایک **خلاصہ کرنے والے آلے** کا استعمال کر سکتے ہیں تاکہ جمع شدہ سیاق و سباق کو ایک مختصر شکل میں سمو دیا جائے۔ ایجنٹ اس آلے کو اہم ترجیحات ریکارڈ کرنے کے لیے بلاتا ہے تاکہ حتیٰ کہ پرانے پیغامات ہٹائے جانے کے باوجود، ضروری معلومات محفوظ رہیں۔

یہ نمونہ زیادہ پیچیدہ تاریخ کی کم کرنے کے لیے بنیادی بلاک ہے:
1. ایجنٹ گفتگو سے اہم حقائق کی شناخت کرتا ہے
2. وہ انہیں برقرار رکھنے کے لیے خلاصہ کرنے والے آلے کو بلاتا ہے
3. پرانے پیغامات کو محفوظ طریقے سے ہٹایا جا سکتا ہے کیونکہ خلاصہ وہ باتیں محفوظ کرتا ہے جو اہم ہیں

نیچے ہم `summarize_preferences` آلے کی تعریف کرتے ہیں جسے ایجنٹ استعمال کر سکتا ہے تاکہ اس نے جو کچھ سیکھا ہے اس کا مختصر خلاصہ ریکارڈ کیا جا سکے۔


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## خلاصہ

اس سبق میں آپ نے Microsoft Agent Framework کا استعمال کرتے ہوئے طویل مدتی ایجنٹ بات چیت میں سیاق و سباق کو کیسے سنبھالنا ہے سیکھا:

### اہم تصورات
- **سیاق و سباق کی ونڈوز محدود ہوتی ہیں** — گفتگو کی تاریخ میں ہر ٹوکن کی قیمت ہوتی ہے اور یہ حد کے تحت شمار ہوتا ہے۔
- **خلاصہ سازی کے اوزار** ایجنٹ کو جمع شدہ سیاق و سباق کو مختصر خلاصوں میں تبدیل کرنے دیتے ہیں، جو ٹوکن کی مقدار کم کرتے ہیں جبکہ اہم معلومات کو برقرار رکھتے ہیں۔
- **ایجنٹ سکرچ پیڈز** مستقل بیرونی یادداشت فراہم کرتے ہیں جو کسی بھی گفتگو کی کمی کے بعد بھی باقی رہتی ہے۔

### آپ نے کیا بنایا
- ایک **سیاق و سباق سے آگاہ ایجنٹ** جو کثیر مرحلوں کی گفتگو میں تسلسل برقرار رکھتا ہے
- ایک **خلاصہ سازی کا آلہ** (`summarize_preferences`) جو صارف کی اہم تفصیلات کو مختصر شکل میں ریکارڈ کرتا ہے
- ایک **کثیر مرحلوں کی گفتگو** جو سیاق و سباق کو برقرار رکھنے اور تبدیلی کے انتظام کی نمائندگی کرتی ہے

### حقیقی دنیا کی درخواستیں
- **کسٹمر سروس بوٹس**: طویل مدد کے سیشنز میں ترجیحات یاد رکھیں
- **ذاتی معاونین**: جاری منصوبوں کو بغیر سیاق و سباق دوبارہ بیان کیے ٹریک کریں
- **تعلیمی اساتذہ**: متعدد بات چیتوں کے دوران طلباء کی پیشرفت برقرار رکھیں

### اگلے اقدامات
- فائل کی بنیاد پر مستقل مزاجی کے ساتھ مکمل سکرچ پیڈ ٹول نافذ کریں
- خلاصہ سازی کے بعد خودکار تاریخ کو کم کریں
- معنوی یادداشت کی تلاش کے لیے ویکٹر ڈیٹا بیس کے ساتھ ملائیں
- ایسے ایجنٹس بنائیں جو کئی دن بعد پوری سیاق و سباق کے ساتھ بات چیت دوبارہ شروع کر سکیں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:  
اس دستاویز کا ترجمہ AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے کیا گیا ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم نوٹ کریں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ سمجھا جانا چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم پر عائد نہیں ہوتی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
